# Lesson 5 — LCEL (LangChain Expression Language)

## Goal

understanding:

- Why LCEL exists.
- What a runnable is.
- How LCEL composes LangChain components.
- The `|` operator.
- Why LCEL replaces repetitive manual pipelines.
- How data flows through an LCEL chain.
- Why almost every modern LangChain examples uses LCEL.

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [6]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [7]:
# Create the pydantic model
from pydantic import BaseModel, Field

class University (BaseModel):
    """Information about a university"""

    name: str = Field(
        description="The name of the university."
    )  
    country: str = Field(
        description="The name of the country that the university located in."
    )

class Student (BaseModel):
    """Student's information"""
    full_name: str = Field(
        description="The full name of the student."
    )  
    age: int = Field(
        description="The age of the student."
    )  
    major: str = Field(
        description="The major that student specialized in."
    )  
    gpa: float = Field(
        description="The gpa of the student."
    )  
    university: University = Field(
        description="The university that the student is registered in."
    ) 

In [8]:
# Create the parser
from langchain_core.output_parsers import PydanticOutputParser
pydantic_parser = PydanticOutputParser(
    pydantic_object= Student
)

print(pydantic_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"University": {"description": "Information about a university", "properties": {"name": {"description": "The name of the university.", "title": "Name", "type": "string"}, "country": {"description": "The name of the country that the university located in.", "title": "Country", "type": "string"}}, "required": ["name", "country"], "title": "University", "type": "object"}}, "description": "Student's information", "properties": {"full_name": {"description": "The full name of the student.", "title": "Full Name", "type": "string"}, "age": {

In [11]:
# Build the prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            {format_instructions}
            """
        ),
        ("human", "{question}")
    ]
).partial(format_instructions = pydantic_parser.get_format_instructions() )

In [ ]:
# Build the chain
chain = prompt | llm | pydantic_parser

In [ ]:
print(type(chain)) # Type of the chain Runnable itself

<class 'langchain_core.runnables.base.RunnableSequence'>


In [ ]:
result = chain.invoke(
    {
        "question": "Tell me about a fictional university student."
    }
)

In [ ]:
print(type(result))
print(result)

<class '__main__.Student'>
full_name='Aria Vance' age=21 major='Astrophysics' gpa=3.85 university=University(name='Aetheria Institute of Technology', country='Canada')


### `batch()`

In [9]:
# Create the parser
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()


In [12]:
# Build the prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a helpful assistant.
            """
        ),
        ("human", "{question}")
    ]
)

In [13]:
# Build the chain
chain = prompt | llm | str_parser

In [14]:
batch_results = chain.batch(
        [
        {"question": "What is LangChain?"},
        {"question": "What is FastAPI?"},
    ]
)

In [15]:
print(type(batch_results))
print(batch_results)

<class 'list'>
['**LangChain** is an open-source framework designed to simplify the creation of applications using Large Language Models (LLMs) like OpenAI\'s GPT-4, Anthropic\'s Claude, or open-source models like Llama. \n\nIf an LLM is the "brain," LangChain acts as the "nervous system and hands"—it connects the LLM to your data, your APIs, and your application logic.\n\nHere is a breakdown of what LangChain is, why it is important, and how it works.\n\n---\n\n### Why is LangChain needed?\nWhile LLMs are incredibly powerful, they have major limitations out of the box:\n1. **No memory:** They don\'t remember past conversations (they are "stateless").\n2. **No access to real-time data:** They are limited by their training cutoff date.\n3. **No access to private data:** They don\'t know about your company\'s internal PDFs, databases, or emails.\n4. **They can\'t take action:** They can write code to book a flight, but they can\'t actually book the flight for you.\n\nLangChain solves the

### `Stream()`

In [16]:
for chunk in chain.stream(
    {
        "question": "Explain LangChain in one paragraph."
    }
): 
    print(type(chunk))
    print(repr(chunk)) # we use  print(repr()) instead of direct print() bec repr makes whitespace and formatting visible

<class 'langchain_core.messages.base.TextAccessor'>
'**LangChain**'
<class 'langchain_core.messages.base.TextAccessor'>
' is an open-source development framework designed to simplify the creation of applications powered by large language models (LLMs). It'
<class 'langchain_core.messages.base.TextAccessor'>
' acts as a powerful orchestrator, enabling developers to connect LLMs (like GPT-4) to external data sources,'
<class 'langchain_core.messages.base.TextAccessor'>
" APIs, and databases, thereby overcoming the models' limitations regarding real-time information and private data. By providing modular components"
<class 'langchain_core.messages.base.TextAccessor'>
' for "chaining" together prompts, models, and data retrieval steps, as well as built-in memory and autonomous'
<class 'langchain_core.messages.base.TextAccessor'>
' agents, LangChain makes it significantly easier to build sophisticated, context-aware AI applications such as chatbots, search tools,'
<class 'langchain_core.mes

### `ainvoke()`

In [ ]:
async def result_main():
    result = await chain.ainvoke(
        {
            "question": "Tell me about LangChain"
        }
    )
    return(result)

In [24]:
# Run async for Jupyter notebook

result = await result_main()
print(result)

**LangChain** is an open-source framework designed to simplify the creation of applications using Large Language Models (LLMs). 

Launched in late 2022 by Harrison Chase, it has quickly become one of the most popular tools in the AI ecosystem. It acts as the "glue" that connects LLMs (like GPT-4, Claude, or Llama) with external data sources, APIs, and computation.

Here is a comprehensive guide to what LangChain is, how it works, and why it is so popular.

---

### 1. Why do we need LangChain?
While LLMs are incredibly powerful, they have several limitations out of the box:
* **They lack up-to-date information:** They only know what was in their training data.
* **They cannot access private data:** They don't know about your company’s PDFs, databases, or emails.
* **They cannot take action:** They can write code, but they can't run it or browse the web on their own.
* **They lack memory:** By default, each API call to an LLM is independent; they don't remember previous questions.

**La

In [26]:
# Run async for normal python script only
"""
import asyncio

result = asyncio.run(result_main())
print(result)
"""

'\nimport asyncio\n\nresult = asyncio.run(result_main())\nprint(result)\n'